# PaySim Fraud Baseline 6D

This notebook is the canonical plaintext baseline for the privacy-preserving fraud detection prototype based on PaySim.

The goal is a compact, stable, HE-friendly inference pipeline rather than state-of-the-art fraud detection. We keep the 6D base representation fixed and compare a small set of linear and low-degree polynomial models that are practical for later HE experiments.


## 1. Project Overview

We focus on a clean plaintext baseline that:

- keeps only `TRANSFER` and `CASH_OUT`
- retains all fraud rows and downsamples non-fraud rows
- uses the same canonical 6D HE-friendly base feature vector
- compares four models on the same sampled split
- exports only the artifacts needed for later HE-side inference

Models in this notebook:

- Ridge score baseline
- Logistic Regression
- LinearSVC
- Degree-2 polynomial Logistic Regression


In [ ]:
from __future__ import annotations

import json
import pickle
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.special import expit
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.svm import LinearSVC

# Ensure project root is on path so `src` is importable
sys.path.insert(0, str(Path(".").resolve()))

from src.config import (
    BINARY_FEATURES,
    CHUNK_SIZE,
    FEATURE_COLUMNS,
    NEG_POS_RATIO,
    NUMERIC_BASE_FEATURES,
    NUMERIC_SCALED_FEATURES,
    RANDOM_STATE,
    SELECTED_TYPES,
    TEST_SIZE,
    VAL_SIZE_WITHIN_TRAIN,
)
from src.data import (
    add_features,
    add_scaled_feature_columns,
    build_sample_dataframe,
    scan_selected_type_counts,
)
from src.evaluation import (
    choose_he_eval_subset,
    evaluate_split,
    maybe_load_previous_baseline,
    summarize_comparison,
)
from src.models import fit_logistic_regression, fit_ridge_score_model

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda value: f"{value:,.4f}")

## 2. Imports And Configuration

The configuration below fixes paths, seeds, selected transaction types, sampling ratio, and the exported artifact set. The base 6D representation and split logic remain unchanged from the current baseline.


In [ ]:
PROJECT_ROOT = Path(".").resolve()
DATA_PATH = PROJECT_ROOT / "data" / "PS_20174392719_1491204439457_log.csv"
ARTIFACT_DIR = PROJECT_ROOT / "artifacts"
ARCHIVE_DIR = PROJECT_ROOT / "archive"

ARTIFACT_DIR.mkdir(exist_ok=True)
ARCHIVE_DIR.mkdir(exist_ok=True)

RANDOM_STATE = 42
CHUNK_SIZE = 250_000
NEG_POS_RATIO = 10
TEST_SIZE = 0.20
VAL_SIZE_WITHIN_TRAIN = 0.20
SELECTED_TYPES = ["TRANSFER", "CASH_OUT"]

NUMERIC_BASE_FEATURES = [
    "amount",
    "oldbalanceOrg",
    "oldbalanceDest",
    "deltaOrig",
    "deltaDest",
]
NUMERIC_SCALED_FEATURES = [
    "amount_scaled",
    "oldbalanceOrg_scaled",
    "oldbalanceDest_scaled",
    "deltaOrig_scaled",
    "deltaDest_scaled",
]
BINARY_FEATURES = ["is_transfer"]
FEATURE_COLUMNS = NUMERIC_SCALED_FEATURES + BINARY_FEATURES

SUMMARY_JSON_PATH = ARTIFACT_DIR / "baseline_6d_summary.json"
COMPARISON_JSON_PATH = ARTIFACT_DIR / "baseline_model_comparison_summary.json"
FEATURE_ORDER_JSON_PATH = ARTIFACT_DIR / "feature_order_6d.json"
SCALER_PICKLE_PATH = ARTIFACT_DIR / "baseline_6d_scaler.pkl"
RIDGE_PICKLE_PATH = ARTIFACT_DIR / "baseline_6d_ridge.pkl"
LOGREG_PICKLE_PATH = ARTIFACT_DIR / "baseline_6d_logreg.pkl"
LINEAR_SVC_PICKLE_PATH = ARTIFACT_DIR / "linear_svc_model.pkl"
POLY2_LOGREG_PICKLE_PATH = ARTIFACT_DIR / "poly2_logreg_model.pkl"
POLY2_TRANSFORMER_PICKLE_PATH = ARTIFACT_DIR / "poly2_transformer.pkl"
HE_FEATURES_PATH = ARTIFACT_DIR / "he_eval_scaled_features_6d.csv"
HE_META_PATH = ARTIFACT_DIR / "he_eval_meta_6d.csv"
HE_SCORES_PATH = ARTIFACT_DIR / "he_eval_plaintext_scores_6d.csv"
PREVIOUS_8D_ARTIFACT_PATH = ARCHIVE_DIR / "baseline_model_artifacts_8d.json"

assert DATA_PATH.exists(), f"Missing dataset: {DATA_PATH}"

print("Project root:", PROJECT_ROOT)
print("Dataset:", DATA_PATH)
print("Artifacts:", ARTIFACT_DIR)
print("Base feature order:", FEATURE_COLUMNS)


## 3. Data Loading And Sampling

We scan the full PaySim CSV in chunks, keep only `TRANSFER` and `CASH_OUT`, retain every fraud row, and downsample non-fraud rows to roughly a 10:1 non-fraud to fraud ratio.


In [ ]:
fraud_count, selected_non_fraud_count = scan_selected_type_counts(DATA_PATH)
target_negative_samples = NEG_POS_RATIO * fraud_count
negative_sample_prob = min(1.0, target_negative_samples / selected_non_fraud_count)

sample_df = build_sample_dataframe(DATA_PATH, negative_sample_prob=negative_sample_prob)

sample_summary_df = pd.DataFrame(
    {
        "metric": [
            "selected_fraud_rows",
            "selected_non_fraud_rows",
            "target_negative_samples",
            "negative_sample_probability",
            "sample_rows",
            "sample_fraud",
            "sample_non_fraud",
        ],
        "value": [
            fraud_count,
            selected_non_fraud_count,
            target_negative_samples,
            negative_sample_prob,
            len(sample_df),
            int(sample_df["isFraud"].sum()),
            int((sample_df["isFraud"] == 0).sum()),
        ],
    }
)

display(sample_summary_df)
display(pd.crosstab(sample_df["type"], sample_df["isFraud"]))

## 4. Feature Engineering

The canonical base representation remains the same:

1. `amount_scaled`
2. `oldbalanceOrg_scaled`
3. `oldbalanceDest_scaled`
4. `deltaOrig_scaled`
5. `deltaDest_scaled`
6. `is_transfer`

Raw engineered features:

- `deltaOrig = oldbalanceOrg - newbalanceOrig`
- `deltaDest = newbalanceDest - oldbalanceDest`
- `is_transfer = 1` for `TRANSFER`, else `0`


In [ ]:
sample_df = add_features(sample_df)
display(sample_df[NUMERIC_BASE_FEATURES + BINARY_FEATURES + ["isFraud"]].head())

## 5. Train / Validation / Test Split

We keep the same deterministic stratified split logic so that all four models are compared on the same sampled rows.


In [ ]:
train_full_df, test_df = train_test_split(
    sample_df,
    test_size=TEST_SIZE,
    stratify=sample_df["isFraud"],
    random_state=RANDOM_STATE,
)

train_df, val_df = train_test_split(
    train_full_df,
    test_size=VAL_SIZE_WITHIN_TRAIN,
    stratify=train_full_df["isFraud"],
    random_state=RANDOM_STATE,
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

split_summary_df = pd.DataFrame(
    [
        {
            "split": "train",
            "rows": len(train_df),
            "fraud": int(train_df["isFraud"].sum()),
            "non_fraud": int((train_df["isFraud"] == 0).sum()),
        },
        {
            "split": "validation",
            "rows": len(val_df),
            "fraud": int(val_df["isFraud"].sum()),
            "non_fraud": int((val_df["isFraud"] == 0).sum()),
        },
        {
            "split": "test",
            "rows": len(test_df),
            "fraud": int(test_df["isFraud"].sum()),
            "non_fraud": int((test_df["isFraud"] == 0).sum()),
        },
    ]
)

display(split_summary_df)


## 6. Scaling

We fit `StandardScaler` on the training split only for the five numeric features:

- `amount`
- `oldbalanceOrg`
- `oldbalanceDest`
- `deltaOrig`
- `deltaDest`

The binary `is_transfer` feature stays unscaled.


In [ ]:
scaler = StandardScaler()
scaler.fit(train_df[NUMERIC_BASE_FEATURES])

train_df = add_scaled_feature_columns(train_df, scaler)
val_df = add_scaled_feature_columns(val_df, scaler)
test_df = add_scaled_feature_columns(test_df, scaler)

X_train = train_df[FEATURE_COLUMNS].to_numpy(dtype=float)
y_train = train_df["isFraud"].to_numpy(dtype=int)
X_val = val_df[FEATURE_COLUMNS].to_numpy(dtype=float)
y_val = val_df["isFraud"].to_numpy(dtype=int)
X_test = test_df[FEATURE_COLUMNS].to_numpy(dtype=float)
y_test = test_df["isFraud"].to_numpy(dtype=int)

display(train_df[FEATURE_COLUMNS + ["isFraud"]].head())

## 7. Model Training

We train four models on the same base 6D representation:

- Ridge score baseline on the 6D inputs
- Logistic Regression on the 6D inputs
- LinearSVC on the 6D inputs
- Degree-2 polynomial Logistic Regression on polynomially expanded 6D inputs

For the polynomial model, we explicitly apply `PolynomialFeatures(degree=2, include_bias=False)` to the scaled 6D base inputs. With 6 base features, the expanded dimension is 27.


In [ ]:
ridge_w, ridge_b = fit_ridge_score_model(X_train, 2 * y_train - 1, alpha=1.0)
logreg_w, logreg_b, logreg_result = fit_logistic_regression(X_train, y_train, l2=1.0, max_iter=500)

linear_svc_model = LinearSVC(
    C=1.0,
    max_iter=5000,
    random_state=RANDOM_STATE,
    dual=False,
)
linear_svc_model.fit(X_train, y_train)

poly2_transformer = PolynomialFeatures(degree=2, include_bias=False)
X_train_poly = poly2_transformer.fit_transform(X_train)
X_val_poly = poly2_transformer.transform(X_val)
X_test_poly = poly2_transformer.transform(X_test)

poly2_logreg_model = LogisticRegression(
    penalty="l2",
    C=1.0,
    solver="lbfgs",
    max_iter=1000,
    random_state=RANDOM_STATE,
)
poly2_logreg_model.fit(X_train_poly, y_train)

poly2_feature_names = poly2_transformer.get_feature_names_out(FEATURE_COLUMNS)
poly2_feature_dim = int(X_train_poly.shape[1])

print("Logistic optimizer success:", bool(logreg_result.success))
print("Logistic optimizer message:", str(logreg_result.message))
print("Polynomial degree-2 expanded feature dimension:", poly2_feature_dim)
display(pd.DataFrame({"poly2_feature_name": poly2_feature_names[:15]}))

## 8. Evaluation

We report precision, recall, F1, PR AUC, confusion matrix, and predicted positive counts for validation and test across all four models.

If the archived 8D artifact is available, we also compare the new 6D results against the previous 8D setup. The main interpretation focus is whether compact 6D inputs remain competitive while staying HE-friendly.


In [ ]:
ridge_val_score = X_val @ ridge_w + ridge_b
ridge_test_score = X_test @ ridge_w + ridge_b
ridge_val_pred = (ridge_val_score >= 0.0).astype(int)
ridge_test_pred = (ridge_test_score >= 0.0).astype(int)

logreg_val_logit = X_val @ logreg_w + logreg_b
logreg_test_logit = X_test @ logreg_w + logreg_b
logreg_val_prob = expit(logreg_val_logit)
logreg_test_prob = expit(logreg_test_logit)
logreg_val_pred = (logreg_val_prob >= 0.5).astype(int)
logreg_test_pred = (logreg_test_prob >= 0.5).astype(int)

linear_svc_val_score = linear_svc_model.decision_function(X_val)
linear_svc_test_score = linear_svc_model.decision_function(X_test)
linear_svc_val_pred = (linear_svc_val_score >= 0.0).astype(int)
linear_svc_test_pred = (linear_svc_test_score >= 0.0).astype(int)

poly2_val_logit = poly2_logreg_model.decision_function(X_val_poly)
poly2_test_logit = poly2_logreg_model.decision_function(X_test_poly)
poly2_val_prob = poly2_logreg_model.predict_proba(X_val_poly)[:, 1]
poly2_test_prob = poly2_logreg_model.predict_proba(X_test_poly)[:, 1]
poly2_val_pred = (poly2_val_prob >= 0.5).astype(int)
poly2_test_pred = (poly2_test_prob >= 0.5).astype(int)

metrics = [
    evaluate_split("validation", "ridge_score", y_val, ridge_val_pred, ridge_val_score),
    evaluate_split("test", "ridge_score", y_test, ridge_test_pred, ridge_test_score),
    evaluate_split("validation", "logistic_regression", y_val, logreg_val_pred, logreg_val_prob),
    evaluate_split("test", "logistic_regression", y_test, logreg_test_pred, logreg_test_prob),
    evaluate_split("validation", "linear_svc", y_val, linear_svc_val_pred, linear_svc_val_score),
    evaluate_split("test", "linear_svc", y_test, linear_svc_test_pred, linear_svc_test_score),
    evaluate_split("validation", "poly2_logistic_regression", y_val, poly2_val_pred, poly2_val_prob),
    evaluate_split("test", "poly2_logistic_regression", y_test, poly2_test_pred, poly2_test_prob),
]

metrics_df = pd.DataFrame(metrics)
model_order = [
    "ridge_score",
    "logistic_regression",
    "linear_svc",
    "poly2_logistic_regression",
]
metrics_df["model"] = pd.Categorical(metrics_df["model"], categories=model_order, ordered=True)
metrics_df["split"] = pd.Categorical(metrics_df["split"], categories=["validation", "test"], ordered=True)
metrics_df = metrics_df.sort_values(["split", "model"]).reset_index(drop=True)
display(metrics_df)

compact_comparison_df = metrics_df[
    ["split", "model", "precision", "recall", "f1", "pr_auc", "predicted_positive"]
].copy()
display(compact_comparison_df)

metrics_by_key = {(item["model"], item["split"]): item for item in metrics}
previous_summary = maybe_load_previous_baseline(PREVIOUS_8D_ARTIFACT_PATH)

if previous_summary is None:
    comparison_to_previous_8d = ["Previous 8D artifact not available in archive/. Comparison skipped."]
else:
    previous_metrics_by_key = {}
    for item in previous_summary.get("metrics", []):
        model_name = item.get("model", "")
        if model_name == "ridge_score_val":
            previous_metrics_by_key[("ridge_score", "validation")] = item
        elif model_name == "ridge_score_test":
            previous_metrics_by_key[("ridge_score", "test")] = item
        elif model_name == "logreg_val":
            previous_metrics_by_key[("logistic_regression", "validation")] = item
        elif model_name == "logreg_test":
            previous_metrics_by_key[("logistic_regression", "test")] = item
    comparison_to_previous_8d = summarize_comparison(previous_metrics_by_key, metrics_by_key)
    if not comparison_to_previous_8d:
        comparison_to_previous_8d = ["Previous 8D artifact found, but comparable metric entries were missing."]

print("Comparison to previous 8D baseline:")
for line in comparison_to_previous_8d:
    print("-", line)

## 9. Artifact Export

We preserve the current canonical 6D artifacts and extend them with the new models:

- `baseline_6d_summary.json`
- `baseline_model_comparison_summary.json`
- `feature_order_6d.json`
- `baseline_6d_scaler.pkl`
- `baseline_6d_ridge.pkl`
- `baseline_6d_logreg.pkl`
- `linear_svc_model.pkl`
- `poly2_logreg_model.pkl`
- `poly2_transformer.pkl`
- `he_eval_scaled_features_6d.csv`
- `he_eval_meta_6d.csv`
- `he_eval_plaintext_scores_6d.csv`

For the polynomial model, the current HE design uses **pre-expanded plaintext polynomial features encrypted afterward**. This avoids encrypted-space polynomial term construction in this pass.


In [ ]:
feature_order = {
    "feature_columns": FEATURE_COLUMNS,
    "numeric_base_features": NUMERIC_BASE_FEATURES,
    "binary_features": BINARY_FEATURES,
}

model_metadata = [
    {
        "model_name": "ridge_score",
        "feature_representation": "base_6d_scaled",
        "score_type": "decision_function",
        "he_ready": True,
        "he_strategy": "encrypt base 6D scaled features and evaluate a weighted sum",
        "coefficient_shape": list(np.asarray(ridge_w).shape),
        "intercept_shape": [],
    },
    {
        "model_name": "logistic_regression",
        "feature_representation": "base_6d_scaled",
        "score_type": "raw_logit",
        "he_ready": True,
        "he_strategy": "encrypt base 6D scaled features and evaluate a weighted sum",
        "coefficient_shape": list(np.asarray(logreg_w).shape),
        "intercept_shape": [],
    },
    {
        "model_name": "linear_svc",
        "feature_representation": "base_6d_scaled",
        "score_type": "decision_function",
        "he_ready": True,
        "he_strategy": "encrypt base 6D scaled features and evaluate a weighted sum",
        "coefficient_shape": list(linear_svc_model.coef_.reshape(-1).shape),
        "intercept_shape": list(linear_svc_model.intercept_.shape),
    },
    {
        "model_name": "poly2_logistic_regression",
        "feature_representation": "poly2_expanded_from_base_6d",
        "score_type": "raw_linear_score_on_poly2_features",
        "he_ready": True,
        "he_strategy": "pre-expand plaintext base 6D features with PolynomialFeatures, then encrypt the expanded vector",
        "coefficient_shape": list(poly2_logreg_model.coef_.reshape(-1).shape),
        "intercept_shape": list(poly2_logreg_model.intercept_.shape),
        "base_feature_order": FEATURE_COLUMNS,
        "expanded_feature_dimension": poly2_feature_dim,
    },
]

he_eval_df = choose_he_eval_subset(test_df)
he_eval_scaled_features_df = he_eval_df[FEATURE_COLUMNS].copy()
he_eval_poly2_features = poly2_transformer.transform(he_eval_scaled_features_df.to_numpy(dtype=float))
he_eval_X = he_eval_scaled_features_df.to_numpy(dtype=float)
he_eval_meta_df = he_eval_df[["sample_row_id", "step", "type", "isFraud"]].copy()
he_eval_scores_df = pd.DataFrame(
    {
        "sample_row_id": he_eval_df["sample_row_id"],
        "isFraud": he_eval_df["isFraud"],
        "ridge_raw_score": he_eval_X @ ridge_w + ridge_b,
        "logistic_raw_logit": he_eval_X @ logreg_w + logreg_b,
        "logistic_probability": expit(he_eval_X @ logreg_w + logreg_b),
        "linear_svc_decision": linear_svc_model.decision_function(he_eval_X),
        "poly2_logreg_raw_score": poly2_logreg_model.decision_function(he_eval_poly2_features),
        "poly2_logreg_probability": poly2_logreg_model.predict_proba(he_eval_poly2_features)[:, 1],
    }
)
detailed_summary = {
    "feature_order": feature_order,
    "scaler": {"mean": scaler.mean_.tolist(), "scale": scaler.scale_.tolist()},
    "ridge_score": {"weights": ridge_w.tolist(), "bias": float(ridge_b)},
    "logistic_regression": {
        "weights": logreg_w.tolist(), "bias": float(logreg_b),
        "optimizer_success": bool(logreg_result.success),
        "optimizer_message": str(logreg_result.message),
    },
    "linear_svc": {
        "weights": linear_svc_model.coef_.reshape(-1).tolist(),
        "bias": float(linear_svc_model.intercept_[0]),
        "classes": linear_svc_model.classes_.tolist(),
    },
    "poly2_logistic_regression": {
        "weights": poly2_logreg_model.coef_.reshape(-1).tolist(),
        "bias": float(poly2_logreg_model.intercept_[0]),
        "expanded_feature_dimension": poly2_feature_dim,
        "optimizer_iterations": poly2_logreg_model.n_iter_.tolist(),
    },
    "poly2_features": {
        "degree": 2, "include_bias": False,
        "expanded_feature_dimension": poly2_feature_dim,
        "base_feature_order": FEATURE_COLUMNS,
        "feature_names": poly2_feature_names.tolist(),
        "he_strategy": "pre-expanded plaintext features encrypted afterward",
    },
    "sample_summary": {
        "rows": int(len(sample_df)),
        "fraud": int(sample_df["isFraud"].sum()),
        "non_fraud": int((sample_df["isFraud"] == 0).sum()),
    },
    "split_summary": {
        "train_rows": int(len(train_df)), "validation_rows": int(len(val_df)), "test_rows": int(len(test_df)),
        "train_fraud": int(train_df["isFraud"].sum()),
        "validation_fraud": int(val_df["isFraud"].sum()),
        "test_fraud": int(test_df["isFraud"].sum()),
    },
    "model_metadata": model_metadata,
    "metrics": metrics,
    "he_eval_subset": {
        "rows": int(len(he_eval_df)),
        "fraud": int(he_eval_df["isFraud"].sum()),
        "non_fraud": int((he_eval_df["isFraud"] == 0).sum()),
    },
}

compact_summary = {
    "base_feature_order": FEATURE_COLUMNS,
    "poly2_expanded_feature_dimension": poly2_feature_dim,
    "models": model_metadata,
    "metrics": metrics,
}

FEATURE_ORDER_JSON_PATH.write_text(json.dumps(feature_order, indent=2))
SUMMARY_JSON_PATH.write_text(json.dumps(detailed_summary, indent=2))
COMPARISON_JSON_PATH.write_text(json.dumps(compact_summary, indent=2))

with SCALER_PICKLE_PATH.open("wb") as f:
    pickle.dump(scaler, f)
with RIDGE_PICKLE_PATH.open("wb") as f:
    pickle.dump({"weights": ridge_w, "bias": ridge_b}, f)
with LOGREG_PICKLE_PATH.open("wb") as f:
    pickle.dump({"weights": logreg_w, "bias": logreg_b}, f)
with LINEAR_SVC_PICKLE_PATH.open("wb") as f:
    pickle.dump(linear_svc_model, f)
with POLY2_LOGREG_PICKLE_PATH.open("wb") as f:
    pickle.dump(poly2_logreg_model, f)
with POLY2_TRANSFORMER_PICKLE_PATH.open("wb") as f:
    pickle.dump(poly2_transformer, f)

he_eval_scaled_features_df.to_csv(HE_FEATURES_PATH, index=False)
he_eval_meta_df.to_csv(HE_META_PATH, index=False)
he_eval_scores_df.to_csv(HE_SCORES_PATH, index=False)

exported_paths = [
    SUMMARY_JSON_PATH, COMPARISON_JSON_PATH, FEATURE_ORDER_JSON_PATH,
    SCALER_PICKLE_PATH, RIDGE_PICKLE_PATH, LOGREG_PICKLE_PATH,
    LINEAR_SVC_PICKLE_PATH, POLY2_LOGREG_PICKLE_PATH, POLY2_TRANSFORMER_PICKLE_PATH,
    HE_FEATURES_PATH, HE_META_PATH, HE_SCORES_PATH,
]
print("Exported baseline artifacts:")
for p in exported_paths:
    print("-", p)


## 10. HE Evaluation Subset Explanation

The HE subset files play the following roles:

- `he_eval_scaled_features_6d.csv` contains the final scaled 6D feature vectors in exact base model input order
- `he_eval_meta_6d.csv` contains labels and lightweight metadata for the same rows
- `he_eval_plaintext_scores_6d.csv` contains plaintext reference scores for all exported models

For the polynomial model, the HE notebook will transform the base 6D features into polynomial degree-2 features in plaintext first, then encrypt the expanded feature vector for score computation.


In [ ]:
final_summary_lines = [
    f"Sample rows: {len(sample_df):,}",
    f"Fraud rows: {int(sample_df['isFraud'].sum()):,}",
    f"Non-fraud rows: {int((sample_df['isFraud'] == 0).sum()):,}",
    f"Final 6D features: {FEATURE_COLUMNS}",
    f"Polynomial degree-2 expanded dimension: {poly2_feature_dim}",
    "",
    "Validation and test metrics:",
]

for metric in metrics_df.to_dict(orient="records"):
    final_summary_lines.append(
        f"{metric['split']} {metric['model']}: "
        f"precision={metric['precision']:.4f}, "
        f"recall={metric['recall']:.4f}, "
        f"f1={metric['f1']:.4f}, "
        f"pr_auc={metric['pr_auc']:.4f}, "
        f"predicted_positive={metric['predicted_positive']}, "
        f"confusion_matrix={metric['confusion_matrix']}"
    )

final_summary_lines.extend(
    [
        "",
        "Exported files:",
        *[f"- {path}" for path in exported_paths],
        "",
        "Ready for next HE stage: yes. The canonical artifacts now cover ridge, logistic regression, LinearSVC, and degree-2 polynomial logistic regression.",
    ]
)

print("\n".join(final_summary_lines))


## 11. Sampling Ratio Sweep

We sweep `NEG_POS_RATIO` over `[5, 10, 20, 30]` and evaluate LogisticRegression and Poly2LogReg on each resulting dataset.
This experiment shows whether adding more non-fraud data improves performance or if the models plateau quickly.
Results are exported to `artifacts/ratio_sweep_results.json` for later visualization.


In [ ]:
RATIO_SWEEP_VALUES = [5, 10, 20, 30]
ratio_sweep_results = []

for _ratio in RATIO_SWEEP_VALUES:
    _neg_prob = min(1.0, _ratio * fraud_count / selected_non_fraud_count)
    _df = build_sample_dataframe(DATA_PATH, negative_sample_prob=_neg_prob)
    _df = add_features(_df)

    _train_full, _test = train_test_split(
        _df, test_size=TEST_SIZE, stratify=_df["isFraud"], random_state=RANDOM_STATE
    )
    _train, _val = train_test_split(
        _train_full, test_size=VAL_SIZE_WITHIN_TRAIN,
        stratify=_train_full["isFraud"], random_state=RANDOM_STATE
    )

    _scaler = StandardScaler()
    _scaler.fit(_train[NUMERIC_BASE_FEATURES])
    _train = add_scaled_feature_columns(_train, _scaler)
    _test  = add_scaled_feature_columns(_test,  _scaler)

    _Xtr = _train[FEATURE_COLUMNS].to_numpy(float)
    _ytr = _train["isFraud"].to_numpy(int)
    _Xte = _test[FEATURE_COLUMNS].to_numpy(float)
    _yte = _test["isFraud"].to_numpy(int)

    # LogReg (custom)
    _w, _b, _ = fit_logistic_regression(_Xtr, _ytr, l2=1.0)
    _prob = expit(_Xte @ _w + _b)
    _pred = (_prob >= 0.5).astype(int)
    _m = evaluate_split("test", "logistic_regression", _yte, _pred, _prob)
    ratio_sweep_results.append({"ratio": _ratio, "train_rows": len(_train), **_m})

    # Poly2LogReg
    _poly = PolynomialFeatures(degree=2, include_bias=False)
    _Xtr_p = _poly.fit_transform(_Xtr)
    _Xte_p = _poly.transform(_Xte)
    _p2 = LogisticRegression(C=1.0, solver="lbfgs", max_iter=1000, random_state=RANDOM_STATE)
    _p2.fit(_Xtr_p, _ytr)
    _prob_p = _p2.predict_proba(_Xte_p)[:, 1]
    _pred_p = (_prob_p >= 0.5).astype(int)
    _m2 = evaluate_split("test", "poly2_logistic_regression", _yte, _pred_p, _prob_p)
    ratio_sweep_results.append({"ratio": _ratio, "train_rows": len(_train), **_m2})

    print(f"ratio={_ratio}:1  rows={len(_df):,}  "
          f"LogReg F1={_m['f1']:.4f}  Poly2 F1={_m2['f1']:.4f}")

ratio_sweep_df = pd.DataFrame(ratio_sweep_results)
display(ratio_sweep_df[["ratio", "model", "train_rows", "f1", "pr_auc"]])

with (ARTIFACT_DIR / "ratio_sweep_results.json").open("w") as _f:
    json.dump(ratio_sweep_results, _f, indent=2)
print("Saved ratio_sweep_results.json")

## 12. Hyperparameter Grid Search

We sweep key regularization parameters for each model:
- **LinearSVC** and **Poly2LogReg**: `GridSearchCV` (3-fold, scoring=F1) over `C ∈ [0.01, 0.1, 1, 10]`
- **Ridge (custom)** and **LogReg (custom)**: manual loop over alpha/l2, pick best validation F1

Best params and resulting test metrics are saved to `artifacts/grid_search_results.json`.


In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import f1_score as _f1_score

C_GRID = [0.01, 0.1, 1, 10]

# LinearSVC
gs_svc = GridSearchCV(
    LinearSVC(max_iter=5000, dual=False, random_state=RANDOM_STATE),
    {"C": C_GRID}, cv=3, scoring="f1", n_jobs=-1, refit=True,
)
gs_svc.fit(X_train, y_train)
best_C_svc = gs_svc.best_params_["C"]
best_svc = gs_svc.best_estimator_
svc_test_score = best_svc.decision_function(X_test)
svc_test_pred  = (svc_test_score >= 0.0).astype(int)
gs_svc_metrics = evaluate_split("test", "linear_svc_tuned", y_test, svc_test_pred, svc_test_score)

# Poly2LogReg
gs_p2 = GridSearchCV(
    LogisticRegression(solver="lbfgs", max_iter=1000, random_state=RANDOM_STATE),
    {"C": C_GRID}, cv=3, scoring="f1", n_jobs=-1, refit=True,
)
gs_p2.fit(X_train_poly, y_train)
best_C_p2 = gs_p2.best_params_["C"]
best_p2 = gs_p2.best_estimator_
p2_test_prob = best_p2.predict_proba(X_test_poly)[:, 1]
p2_test_pred = (p2_test_prob >= 0.5).astype(int)
gs_p2_metrics = evaluate_split("test", "poly2_logreg_tuned", y_test, p2_test_pred, p2_test_prob)

# Ridge (custom) — manual val loop
best_ridge_alpha, best_ridge_val_f1 = 1.0, -1.0
for _alpha in [0.01, 0.1, 1, 10, 100]:
    _w, _b = fit_ridge_score_model(X_train, 2 * y_train - 1, alpha=_alpha)
    _pred_v = (X_val @ _w + _b >= 0.0).astype(int)
    _f1 = _f1_score(y_val, _pred_v, zero_division=0)
    if _f1 > best_ridge_val_f1:
        best_ridge_val_f1, best_ridge_alpha = _f1, _alpha
best_ridge_w, best_ridge_b = fit_ridge_score_model(X_train, 2 * y_train - 1, alpha=best_ridge_alpha)
ridge_tuned_score = X_test @ best_ridge_w + best_ridge_b
ridge_tuned_pred  = (ridge_tuned_score >= 0.0).astype(int)
gs_ridge_metrics  = evaluate_split("test", "ridge_tuned", y_test, ridge_tuned_pred, ridge_tuned_score)

# LogReg (custom) — manual val loop
best_logreg_l2, best_logreg_val_f1 = 1.0, -1.0
for _l2 in [0.01, 0.1, 1, 10]:
    _w, _b, _ = fit_logistic_regression(X_train, y_train, l2=_l2)
    _pred_v = (expit(X_val @ _w + _b) >= 0.5).astype(int)
    _f1 = _f1_score(y_val, _pred_v, zero_division=0)
    if _f1 > best_logreg_val_f1:
        best_logreg_val_f1, best_logreg_l2 = _f1, _l2
best_logreg_w, best_logreg_b, _ = fit_logistic_regression(X_train, y_train, l2=best_logreg_l2)
logreg_tuned_prob = expit(X_test @ best_logreg_w + best_logreg_b)
logreg_tuned_pred = (logreg_tuned_prob >= 0.5).astype(int)
gs_logreg_metrics = evaluate_split("test", "logreg_tuned", y_test, logreg_tuned_pred, logreg_tuned_prob)

grid_search_results = {
    "ridge":    {"best_alpha": best_ridge_alpha,  "metrics": gs_ridge_metrics},
    "logreg":   {"best_l2":    best_logreg_l2,    "metrics": gs_logreg_metrics},
    "linear_svc": {"best_C":  best_C_svc,         "metrics": gs_svc_metrics,
                   "cv_results": {k: v.tolist() if hasattr(v, 'tolist') else list(v)
                                  for k, v in gs_svc.cv_results_.items()
                                  if k in ('params', 'mean_test_score', 'std_test_score')}},
    "poly2_logreg": {"best_C": best_C_p2,         "metrics": gs_p2_metrics,
                     "cv_results": {k: v.tolist() if hasattr(v, 'tolist') else list(v)
                                    for k, v in gs_p2.cv_results_.items()
                                    if k in ('params', 'mean_test_score', 'std_test_score')}},
}

with (ARTIFACT_DIR / "grid_search_results.json").open("w") as _f:
    json.dump(grid_search_results, _f, indent=2)

gs_summary_df = pd.DataFrame([
    {"model": "ridge",        "best_param": f"alpha={best_ridge_alpha}",  **gs_ridge_metrics},
    {"model": "logreg",       "best_param": f"l2={best_logreg_l2}",       **gs_logreg_metrics},
    {"model": "linear_svc",   "best_param": f"C={best_C_svc}",            **gs_svc_metrics},
    {"model": "poly2_logreg", "best_param": f"C={best_C_p2}",             **gs_p2_metrics},
])
display(gs_summary_df[["model", "best_param", "precision", "recall", "f1", "pr_auc"]])
print("Saved grid_search_results.json")

## 13. Non-HE Baselines and Test Score Export

We add `RandomForestClassifier` and `GradientBoostingClassifier` as upper-bound baselines.
These are **not HE-compatible** but provide a ceiling for comparison on the poster.

We also export all model scores on the full test set for use in `visualization.ipynb`.


In [ ]:
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier

rf_model = RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1)
rf_model.fit(X_train, y_train)
rf_test_prob = rf_model.predict_proba(X_test)[:, 1]
rf_test_pred = (rf_test_prob >= 0.5).astype(int)
rf_metrics   = evaluate_split("test", "random_forest", y_test, rf_test_pred, rf_test_prob)

gb_model = GradientBoostingClassifier(n_estimators=100, random_state=RANDOM_STATE)
gb_model.fit(X_train, y_train)
gb_test_prob = gb_model.predict_proba(X_test)[:, 1]
gb_test_pred = (gb_test_prob >= 0.5).astype(int)
gb_metrics   = evaluate_split("test", "gradient_boosting", y_test, gb_test_pred, gb_test_prob)

# Comparison table: HE models (baseline) + tuned + non-HE
extended_rows = [
    {"model": "ridge",              "he_compatible": True,  **next(m for m in metrics if m["model"]=="ridge_score" and m["split"]=="test")},
    {"model": "logistic_regression", "he_compatible": True,  **next(m for m in metrics if m["model"]=="logistic_regression" and m["split"]=="test")},
    {"model": "linear_svc",          "he_compatible": True,  **next(m for m in metrics if m["model"]=="linear_svc" and m["split"]=="test")},
    {"model": "poly2_logreg",        "he_compatible": True,  **next(m for m in metrics if m["model"]=="poly2_logistic_regression" and m["split"]=="test")},
    {"model": "ridge_tuned",         "he_compatible": True,  **gs_ridge_metrics},
    {"model": "logreg_tuned",        "he_compatible": True,  **gs_logreg_metrics},
    {"model": "linear_svc_tuned",    "he_compatible": True,  **gs_svc_metrics},
    {"model": "poly2_logreg_tuned",  "he_compatible": True,  **gs_p2_metrics},
    {"model": "random_forest",       "he_compatible": False, **rf_metrics},
    {"model": "gradient_boosting",   "he_compatible": False, **gb_metrics},
]
extended_df = pd.DataFrame(extended_rows)
display(extended_df[["model", "he_compatible", "precision", "recall", "f1", "pr_auc"]])

with (ARTIFACT_DIR / "extended_model_comparison.json").open("w") as _f:
    json.dump(extended_rows, _f, indent=2)

# Export full test set for visualization notebook
test_features_df = test_df[FEATURE_COLUMNS].copy()
test_features_df.to_csv(ARTIFACT_DIR / "test_features_6d.csv", index=False)
pd.DataFrame({"isFraud": y_test}).to_csv(ARTIFACT_DIR / "test_labels.csv", index=False)

test_scores_df = pd.DataFrame({
    "isFraud":               y_test,
    "ridge_score":           X_test @ ridge_w + ridge_b,
    "ridge_tuned_score":     X_test @ best_ridge_w + best_ridge_b,
    "logreg_prob":           expit(X_test @ logreg_w + logreg_b),
    "logreg_tuned_prob":     logreg_tuned_prob,
    "linear_svc_score":      linear_svc_model.decision_function(X_test),
    "linear_svc_tuned_score": svc_test_score,
    "poly2_logreg_prob":     poly2_test_prob,
    "poly2_logreg_tuned_prob": p2_test_prob,
    "random_forest_prob":    rf_test_prob,
    "gradient_boosting_prob": gb_test_prob,
})
test_scores_df.to_csv(ARTIFACT_DIR / "test_scores_6d.csv", index=False)
print("Exported test_features_6d.csv, test_labels.csv, test_scores_6d.csv")
print("Exported extended_model_comparison.json")